<a href="https://colab.research.google.com/github/edwardsnj/glygen-colab-notebooks/blob/main/variants/main1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#
# Import the modules from the github repository
#
import httpimport

with httpimport.github_repo("edwardsnj","glygen-colab-notebooks", ref="main"):
  from glygen import GlyGenDownloader


In [2]:
#
# Parameters for the figure
#

SPECIES = "human"



In [3]:
#
# Make lists of GlyGen reviewed data-files for species-specific glycosites
# using the GlyGenDownloader class
#

ggdl = GlyGenDownloader()

# UniProt is a mix of experimental and predicted
template = "{species}_proteoform_glycosylation_sites_uniprotkb.csv"
uniprotkb_site_files = ggdl.filenames(template,species=SPECIES)

# Some data-files have predicted sites
template = "{species}_proteoform_glycosylation_sites_predicted_*.csv"
pred_site_files = ggdl.filenames(template,species=SPECIES)

# The rest (not uniprot, not predicted) are experimental sites
template = "{species}_proteoform_glycosylation_sites_*.csv"
site_files = ggdl.filenames(template,species=SPECIES)

exp_site_files = sorted(set(site_files)-set(pred_site_files)-set(uniprotkb_site_files))


URLError: <urlopen error [Errno 101] Network is unreachable>

In [ ]:
#
# Use the GlyGenDownloader to construct data-frames for each of these classes
# of GlyGen reviewed data-files. Set the predicted column appropraitely.
#

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa","glycosylation_type"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "setcolumn": {"predicted": False}
}
glyco_site_exp = ggdl.cached_dataframe("glyco_site_exp",exp_site_files,**params)

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa","glycosylation_type"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "setcolumn": {"predicted": True}
}
glyco_site_pred = ggdl.cached_dataframe("glyco_site_pred",pred_site_files,**params)

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa",
              "glycosylation_type","xref_key"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "transform": {"predicted":
                lambda df: df["xref_key"].isin(["protein_xref_pubmed",
                                                "protein_xref_doi"])
                },
  "dropcols": ["xref_key"]
}
glyco_site_uniprotkb = ggdl.cached_dataframe("glyco_site_uniprotkb",uniprotkb_site_files,**params)


In [ ]:
#
# Concatenate the three data-frames, and subset
# on the predicted column, and then remove the
# predicted column
#

import pandas as pd

glyco_sites = pd.concat([glyco_site_exp,
                         glyco_site_pred,
                         glyco_site_uniprotkb],
                        ignore_index=True)
glyco_sites.drop_duplicates(inplace=True)
glyco_sites.reset_index(inplace=True,drop=True)

glyco_site_exp = glyco_sites[~glyco_sites['predicted']]
glyco_site_exp = glyco_site_exp.drop(columns=['predicted'])

glyco_site_pred = glyco_sites[glyco_sites['predicted']]
glyco_site_pred = glyco_site_pred.drop(columns=['predicted'])

glyco_sites_exp.to_csv("data/glyco_sites_exp.csv",index=False)
print(f"Wrote data/glyco_sites_exp.csv: {glyco_sites_exp.shape[0]} rows.")

glyco_sites_pred.to_csv("data/glyco_sites_pred.csv",index=False)
print(f"Wrote data/glyco_sites_pred.csv: {glyco_sites_pred.shape[0]} rows.")


In [ ]:
#
# get mutations data, germline or somatic
#

template = "{species}_protein_mutation_cancer_all.csv"
cancer_files = ggdl.filenames(template,species=SPECIES)

template = "{species}_protein_mutation_germline_all.csv"
germline_files = ggdl.filenames(template,species=SPECIES)

params = {
  "usecols": ["uniprotkb_canonical_ac",
              "aa_pos","ref_aa","alt_aa",
              "do_name"]
  "notna": ["uniprotkb_canonical_ac","aa_pos","ref_aa"],
  "asint": ["aa_pos"],
  "setcolumn": {"variant_type": "somatic_cancer"},
  "transform": {"dstatus": lambda df: ~df["do_name"].isna()},
  "filterrows": [ lambda df: (df["aa_pos"]>1),
                  lambda df: (df["ref_aa"] != df["alt_aa"]) ],
  "dropdups": True,
  "dropcols": ["do_name"]
}
cancer_df = ggdl.cached_dataframe("cancer_df",cancer_files,**params)

params = {
  "usecols": ["uniprotkb_canonical_ac",
              "begin_aa_pos","end_aa_pos","ref_aa","alt_aa",
              "do_id","mim_id"]
  "notna": ["uniprotkb_canonical_ac","aa_pos","ref_aa"],
  "asint": ["begin_aa_pos","end_aa_pos"],
  "setcolumn": {"variant_type": "germline"},
  "transform": {"aa_pos": lambda df: df["begin_aa_pos"],
                "dstatus": lambda df: (~df["do_id"].isna() & ~df["mim_id"].isna()
                },
  "filterrows": [ lambda df: (df["begin_aa_pos"]>1),
                  lambda df: (df["begin_aa_pos"] == df["end_aa_pos"]),
                  lambda df: (df["ref_aa"] != df["alt_aa"]) ],
  "dropdups": True,
  "dropcols": ["begin_aa_pos", "end_aa_pos", "do_id", "mim_id"]
}
germline_df = ggdl.cached_dataframe("germline_df",germline_files,**params)
